# further classifications

In [1]:
import os
from pathlib import Path
import pandas as pd
import duckdb as ddb
from connection_helper import sql
from pandas_plots import tbl, pls, hlp
from pandas_plots.hlp import add_bitmask_label
import duckdb as ddb

hlp.show_package_version(["pygwalker"])
os.environ['THEME']='light'
os.environ['DEBUG']='1'

dir_db=Path("C://temp") if hlp.get_os(hlp.OperatingSystem.WINDOWS) else Path(os.path.expanduser("~/tmp"))

file_db_clin = dir_db/'2025-11-11_data_clin.duckdb'
if not file_db_clin.exists():
    exit()

if not os.path.exists(".local"):
    os.makedirs(".local")

🐍 3.12.8 | 📦 pygwalker: 0.4.9.15 | 📦 pandas: 2.3.3 | 📦 numpy: 1.26.4 | 📦 duckdb: 1.4.2 | 📦 pandas-plots: 0.23.1 | 📦 connection-helper: 0.13.2


In [2]:
con = ddb.connect(file_db_clin, read_only=True)
_=con.execute("PRAGMA disable_progress_bar;")

## <a id='toc1_1_'></a>[📆 data as of](#toc0_)

In [3]:
sql.print_meta(file_db_clin)

database file:           2025-11-11_data_clin.duckdb
data tag:                v2.3
last kkr data import:    2025-09-30
sql table created:       2025-11-11 11:52:01
doi:                     10.18444/5.03.01.0005.0021.0002
document created:        2025-12-02 13:45:53


In [4]:
# # * list all
con.sql("""select distinct(Name) from Diagnose_WeitereKlassifikation order by Name""").to_df().Name.unique()

array(['19q', '1p', '1p19q-Deletion', 'AAIPI', 'ABSTANDPT_MM', 'ADOREG',
       'AEG', 'AEG (Ösophagus)', 'AEG n Siewert', 'AEG nach Siewert',
       'AEG/Siewert (Ösophagus/Kardia)', 'AEG/Siewert-Klassifikation',
       'AJC - Osteosarkome', 'AJC - Weichteile', 'AJC/UICC', 'AJCC',
       'AJCC (Haut)', 'AJCC (kutane T-Zell-Lymphome)',
       'AJCC (nicht Haut)', 'AJCC 2016', 'AJCC Klassifikation MM 2016',
       'AJCC Stadieneinteilung MM 2017', 'AJCC+TNM (Malignes Melanom)',
       'AJCC-STADIUM', 'AJCC-Stadieneinteilung', 'AJCC-malignes Melanom',
       'ALK', 'AML ELN - C92.0', 'AML ELN-Klassifikation (2010)',
       'AML EuropLeukNet 2017', 'AML EuropLeukNet 2017 (ELN)',
       'AML EuropLeukNet 2022', 'AML European Leukemia Net',
       'AML European LeukemiaNet', 'ANN_ARBOR', 'ANN_ARBOR_BULKY',
       'ANN_ARBOR_EXTRA', 'ANN_ARBOR_MILZ', 'ANN_ARBOR_STADIUM',
       'ANN_ARBOR_ZUSATZ', 'AP (alkalische Phosphatase)',
       'ASA Risikoklassifikation',
       'Adenokarzinome des ös

In [5]:
db_class = (con.sql("""--sql
        with dia_fol as (
            select Name, Stadium, z_tum_id,
            'dia' as source 
            from Diagnose_WeitereKlassifikation
            union
            select Name, Stadium, z_tum_id,
            'fo' as source 
            from Folgeereignis_WeitereKlassifikation
        )
        select
            tum.z_tum_id, Name, Stadium, z_kkr_label, source
            ,case 
                when regexp_matches(Name, 'arbor', 'i') then 'ann_arbor'
                when regexp_matches(Name, 'who|gehirn|brain', 'i') then 'brain'
                when regexp_matches(Name, 'strogen', 'i') then 'oestrogen'
                when regexp_matches(Name, 'uicc', 'i') then 'uicc'
                when regexp_matches(Name, 'psa', 'i') then 'psa'
                when regexp_matches(Name, 'breslow', 'i') then 'breslow'
                when regexp_matches(Name, 'gleason', 'i') then 'gleason'
                when regexp_matches(Name, 'p16|hpv', 'i') then 'hpv'
                when regexp_matches(Name, 'hep', 'i') then 'hep' -- M+(HEP) for colorectal cancer
                when regexp_matches(Name, 'ki67|ki-67', 'i') then 'ki67'
            end as class
        from dia_fol
        join Tumor tum on dia_fol.z_tum_id = tum.z_tum_id
        where Stadium is not null
    """)
)
tbl.descr_db(db_class, "db_class")

🗄️ db_class	653_691, 6
	("z_tum_id, Name, Stadium, z_kkr_label, source, class")
┌──────────────────────────────────────┬────────────────────────┬─────────┬─────────────┬─────────┬─────────┐
│               z_tum_id               │          Name          │ Stadium │ z_kkr_label │ source  │  class  │
│               varchar                │        varchar         │ varchar │   varchar   │ varchar │ varchar │
├──────────────────────────────────────┼────────────────────────┼─────────┼─────────────┼─────────┼─────────┤
│ 0e7ba901-8512-4346-933d-019da31093f7 │ PROSTATA.Gleason-Score │ 4+4=8   │ 09-BY       │ fo      │ gleason │
│ c5635830-7173-4786-9724-59eac004e346 │ Gleason-Score          │ 3+4=7a  │ 06-HE       │ fo      │ gleason │
│ a9b806d9-b1d7-468c-a3e8-7f30c6e0deb7 │ Gleason-Score          │ 4+4=8   │ 06-HE       │ fo      │ gleason │
└──────────────────────────────────────┴────────────────────────┴─────────┴─────────────┴─────────┴─────────┘



In [6]:
if os.getenv("DEBUG") == "1":
    db_class.filter("class = 'ki67'").unique("Name")

In [7]:
db_class_agg = (db_class
    .aggregate("z_kkr_label, source, class, count(*) as cnt, count(distinct z_tum_id) as cnt_tum")
)

In [8]:
_df = db_class.aggregate("class, source, count(*) as cnt").to_df()
pls.plot_stacked_bars(_df, orientation="h", sort_values_index=True)

In [10]:
_df = (
    db_class_agg
    # .project("class, source, z_kkr_label")
    .project("class, source, z_kkr_label, cnt")
    .to_df()
    .dropna()
)
_ = pls.plot_facet_stacked_bars(
    _df,
    renderer=None,
    # relative=True,
    annotations=True,
)

In [11]:
_df = db_class_agg.project("z_kkr_label, class, cnt").to_df().dropna()
tbl.pivot_df(
    _df,
    swap=True,
    data_bar_axis="",
    heatmap_axis="xy",
    pct_axis="",
    total_mode="",
)


z_kkr_label,01-SH,02-HH,03-NI,04-HB,05-NW,06-HE,07-RP,08-BW,09-BY,10-SL,11-BE,12-BB,13-MV,14-SN,15-ST,16-TH
class,,,,,,,,,,,,,,,,
ann_arbor,1_685,790,1_905,345,0,3_154,1_032,7_001,6_426,465,2_855,2_344,1_985,4_793,1_584,1_293
brain,1_786,1_282,1_625,362,0,4_150,685,12_360,6_643,746,3_888,3_940,2_446,3_966,1_678,1_420
breslow,13,403,639,400,0,1_212,163,2_941,812,700,1_859,782,300,175,2_772,1_461
gleason,44,307,455,94,0,2_413,710,12_979,8_585,1_578,599,1_279,1_451,4_991,2_306,1_926
hpv,286,1_688,1_923,530,0,3_531,141,10_706,4_829,1_490,3_255,2_302,1_715,2_251,452,737
ki67,4,2_759,283,103,0,1_426,63,19_643,0,29,3,1,1,0,2,3
oestrogen,0,0,3,0,4_487,0,0,3,0,0,0,0,0,0,0,12
psa,0,46,1,0,95_924,0,63,1_099,0,0,9,3,0,0,0,0
uicc,3_622,3_014,26_431,35,0,12_331,1_431,48_925,9_707,3_963,633,545,406,5_714,1_526,10


In [12]:
if False:
    import pygwalker as pyg

    pyg.walk(
        db_class_agg.to_df().dropna(),
        kernel_computation=True,
    )